## 학습 목표

1. **이전 ReAct 과정과 RAG의 연결점**을 이해하고, 프롬프팅 기법의 발전 흐름 속에서 RAG의 위치를 설명할 수 있다.
2. **LLM의 본질적 한계**(지식 컷오프, 환각, 도메인 전문성 부족)를 구체적으로 인식한다.
3. **RAG(Retrieval-Augmented Generation)** 의 핵심 아이디어와 아키텍처를 이해한다.
4. **ReAct와 RAG의 공통점과 차이점** 을 비교 분석할 수 있다.
5. RAG 파이프라인의 3단계(Indexing, Retrieval, Generation) 구조를 파악한다.

## 1. 이전 과정 회고와 RAG로의 전환

이전 ReAct 과정에서 우리는 **Thought-Action-Observation(TAO) 루프** 를 학습하고, LLM이 외부 도구(Search, Calculator 등)를 호출하여 정보를 수집하는 **에이전트 워크플로우** 를 구축했습니다.

ReAct의 핵심 통찰은 다음과 같았습니다:
- LLM은 **내부 지식만으로는 한계** 가 있다
- **외부 도구를 통해 실시간 정보를 수집** 하면 환각을 줄일 수 있다
- 추론(Reasoning)과 행동(Acting)을 **교차 반복** 하여 정확도를 높인다

이번 RAG 과정에서는 이 개념을 **더 체계적이고 구조화된 방향** 으로 확장합니다.

```
프롬프팅 기법의 발전:
Zero-shot → Few-shot → CoT → ReAct → RAG
(단순 질의)  (예시 제공)  (추론 명시)  (추론+행동)  (검색+생성)

ReAct의 Search Tool  ──────→  RAG의 Retrieval 단계
(매번 실시간 검색)            (사전 구축된 지식 베이스에서 검색)
```

**핵심 전환점**: ReAct에서는 Search API를 통해 매번 외부 검색을 수행했지만, RAG에서는 **사전에 문서를 벡터화하여 저장** 해 두고, 질문이 들어오면 **벡터 유사도 기반으로 관련 문서를 빠르게 검색** 합니다. 이를 통해 검색 속도, 정확도, 비용 면에서 큰 이점을 얻습니다.

## 2. LLM의 본질적 한계

대규모 언어 모델(LLM)은 방대한 텍스트 데이터를 학습하여 놀라운 언어 생성 능력을 보여주지만, 본질적으로 다음과 같은 한계를 가지고 있습니다.

### 2.1 지식 컷오프(Knowledge Cutoff)

LLM은 학습 데이터의 시점 이후에 발생한 사건이나 정보를 알지 못합니다. 예를 들어 2024년까지의 데이터로 학습된 모델은 2025년의 사건에 대해 정확한 답변을 제공할 수 없습니다.

### 2.2 환각(Hallucination)

LLM은 학습 데이터에 존재하지 않는 정보를 마치 사실인 것처럼 생성할 수 있습니다. 이는 모델이 확률적으로 그럴듯한 텍스트를 생성하는 방식에서 기인합니다.

### 2.3 도메인 전문성 부족

범용 LLM은 특정 기업의 내부 문서, 전문 분야의 최신 연구, 비공개 데이터베이스의 정보에 접근할 수 없습니다. 도메인 특화된 질문에 대해 일반적이거나 부정확한 답변을 제공할 가능성이 높습니다.

In [1]:
print('''
============================================================
[한계 1] 지식 컷오프(Knowledge Cutoff)
============================================================
  질문: 2025년 노벨 물리학상 수상자는 누구인가요?
  LLM 예상 응답: 학습 데이터 이후의 정보이므로 정확한 답변이 어렵습니다.
  문제점: 학습 데이터의 시점(cutoff) 이후 정보에 접근 불가
  RAG 해결책: 최신 문서를 벡터 DB에 저장하여 검색 후 답변 생성

============================================================
[한계 2] 환각(Hallucination)
============================================================
  질문: XYZ-3000 프로토콜의 주요 특징은 무엇인가요?
  LLM 예상 응답: XYZ-3000은 분산 시스템에서 사용되는... (존재하지 않는 정보를 생성)
  문제점: 존재하지 않는 정보를 사실처럼 생성
  RAG 해결책: 검증된 문서에서만 정보를 검색하여 근거 기반 답변 생성

============================================================
[한계 3] 도메인 전문성 부족
============================================================
  질문: 우리 회사의 내부 보안 정책 중 비밀번호 규칙은?
  LLM 예상 응답: 일반적인 보안 정책을 설명하지만 특정 회사 정보는 없음
  문제점: 비공개/특정 도메인 데이터에 대한 지식 없음
  RAG 해결책: 회사 내부 문서를 벡터 DB에 인덱싱하여 도메인 특화 답변
''')


[한계 1] 지식 컷오프(Knowledge Cutoff)
  질문: 2025년 노벨 물리학상 수상자는 누구인가요?
  LLM 예상 응답: 학습 데이터 이후의 정보이므로 정확한 답변이 어렵습니다.
  문제점: 학습 데이터의 시점(cutoff) 이후 정보에 접근 불가
  RAG 해결책: 최신 문서를 벡터 DB에 저장하여 검색 후 답변 생성

[한계 2] 환각(Hallucination)
  질문: XYZ-3000 프로토콜의 주요 특징은 무엇인가요?
  LLM 예상 응답: XYZ-3000은 분산 시스템에서 사용되는... (존재하지 않는 정보를 생성)
  문제점: 존재하지 않는 정보를 사실처럼 생성
  RAG 해결책: 검증된 문서에서만 정보를 검색하여 근거 기반 답변 생성

[한계 3] 도메인 전문성 부족
  질문: 우리 회사의 내부 보안 정책 중 비밀번호 규칙은?
  LLM 예상 응답: 일반적인 보안 정책을 설명하지만 특정 회사 정보는 없음
  문제점: 비공개/특정 도메인 데이터에 대한 지식 없음
  RAG 해결책: 회사 내부 문서를 벡터 DB에 인덱싱하여 도메인 특화 답변

